In [5]:

#Carregamento de dados
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 13})
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

ARQUIVO = "votacao_candidato_munzona_2026_DF.xlsx"
ABA = "votacao_candidato_munzona_2026_"
# procura o arquivo em data/, na pasta atual ou no Colab (/content)
DATA_PATH = next(p for p in [Path("data")/ARQUIVO, Path(ARQUIVO), Path("/content")/ARQUIVO] if p.exists())

IMG = Path("images"); IMG.mkdir(exist_ok=True)
def salvar(nome): plt.savefig(IMG/f"{nome}.png", bbox_inches="tight", dpi=160)

df = pd.read_excel(DATA_PATH, sheet_name=ABA)
print(f"Linhas: {df.shape[0]:,} | Colunas: {df.shape[1]}")

Linhas: 11,381 | Colunas: 50


In [7]:
#Antes de analisar, valido o que a base realmente contém.
resumo = pd.DataFrame({
    "Eleição":        [df.DS_ELEICAO.iloc[0]],
    "Turno":          [df.NR_TURNO.unique().tolist()],
    "Zonas":          [df.NR_ZONA.nunique()],
    "Candidatos":     [df.SQ_CANDIDATO.nunique()],
    "Partidos":       [df.SG_PARTIDO.nunique()],
    "Nulos (total)":  [int(df.isna().sum().sum())],
    "Duplicados":     [int(df.duplicated().sum())],
}).T.rename(columns={0: "Valor"})
display(resumo)

cargos = (df.groupby("DS_CARGO")
            .agg(candidatos=("SQ_CANDIDATO", "nunique"),
                 votos_nominais=("QT_VOTOS_NOMINAIS", "sum"),
                 votos_validos=("QT_VOTOS_NOMINAIS_VALIDOS", "sum"))
            .sort_values("votos_validos", ascending=False))
cargos["votos_anulados_sub_judice"] = cargos.votos_nominais - cargos.votos_validos
display(cargos)

,Valor
Eleição,ELEIÇÕES GERAIS ESTADUAIS 2026
Turno,[1]
Zonas,19
Candidatos,599
Partidos,29
Nulos (total),0
Duplicados,0


,candidatos,votos_nominais,votos_validos,votos_anulados_sub_judice
DS_CARGO,,,,
Senador,11,3230557,3226319,4238
Governador,10,1653237,1651394,1843
Deputado Distrital,415,1627913,1626978,935
Deputado Federal,163,1615634,1602806,12828


In [8]:
#Ranking dos Candidatos Mais Votados para Governador
VOTOS = "QT_VOTOS_NOMINAIS_VALIDOS"
gov = df[df.DS_CARGO == "Governador"].copy()
sen = df[df.DS_CARGO == "Senador"].copy()
dd  = df[df.DS_CARGO == "Deputado Distrital"].copy()
df_fed = df[df.DS_CARGO == "Deputado Federal"].copy()

def ranking(base, n=None):
    r = (base.groupby(["NM_URNA_CANDIDATO", "SG_PARTIDO"])[VOTOS].sum()
             .sort_values(ascending=False).reset_index())
    r["% votos"] = r[VOTOS] / r[VOTOS].sum() * 100
    r.index = r.index + 1
    return r.head(n) if n else r

ranking(gov, 10)

,NM_URNA_CANDIDATO,SG_PARTIDO,QT_VOTOS_NOMINAIS_VALIDOS,% votos
1,CELINA LEÃO,PP,825530,49.99
2,LEANDRO GRASS,PT,569930,34.51
3,PAULA BELMONTE,PSDB,140765,8.52
4,KIKO CAPUTO,NOVO,72148,4.37
5,CAPPELLI,PSB,36828,2.23
6,PROFESSORA SAMARA MINEIRO,UP,3958,0.24
7,PROFESSOR ROBSON,PSTU,1640,0.10
8,EXPEDITO MENDONÇA,PCO,595,0.04
9,ELISSON,AGIR,0,0.00
10,RICO PINHEIRO,PRTB,0,0.00


In [9]:
# Governador — resultado geral
rk_gov = ranking(gov)
display(rk_gov)

primeiro, segundo = rk_gov.iloc[0], rk_gov.iloc[1]
dif = primeiro[VOTOS] - segundo[VOTOS]
print(f"1º {primeiro.NM_URNA_CANDIDATO}: {primeiro[VOTOS]:,.0f} ({primeiro['% votos']:.1f}%)")
print(f"2º {segundo.NM_URNA_CANDIDATO}: {segundo[VOTOS]:,.0f} ({segundo['% votos']:.1f}%)")


,NM_URNA_CANDIDATO,SG_PARTIDO,QT_VOTOS_NOMINAIS_VALIDOS,% votos
1,CELINA LEÃO,PP,825530,49.99
2,LEANDRO GRASS,PT,569930,34.51
3,PAULA BELMONTE,PSDB,140765,8.52
4,KIKO CAPUTO,NOVO,72148,4.37
5,CAPPELLI,PSB,36828,2.23
6,PROFESSORA SAMARA MINEIRO,UP,3958,0.24
7,PROFESSOR ROBSON,PSTU,1640,0.10
8,EXPEDITO MENDONÇA,PCO,595,0.04
9,ELISSON,AGIR,0,0.00
10,RICO PINHEIRO,PRTB,0,0.00


1º CELINA LEÃO: 825,530 (50.0%)
2º LEANDRO GRASS: 569,930 (34.5%)


In [13]:
#Governador - Vencedor por zona zona eleitoral
zona_cand = (gov.groupby(["NR_ZONA", "NM_URNA_CANDIDATO"])[VOTOS].sum().unstack(fill_value=0))
total_zona = zona_cand.sum(axis=1)
share = zona_cand.div(total_zona, axis=0) * 100

duelo = pd.DataFrame({
    "votos_zona": total_zona,
    "celina_%":   share["CELINA LEÃO"],
    "grass_%":    share["LEANDRO GRASS"],
})
duelo["margem"] = duelo["celina_%"] - duelo["grass_%"]          
duelo["vencedor"]  = zona_cand.idxmax(axis=1)
duelo = duelo.sort_values("margem")
display(duelo)
print(duelo.vencedor.value_counts().to_string())

,votos_zona,celina_%,grass_%,margem,vencedor
NR_ZONA,,,,,
14,85312,36.78,46.75,-9.97,LEANDRO GRASS
1,57662,39.24,42.76,-3.52,LEANDRO GRASS
11,58665,43.92,38.90,5.02,CELINA LEÃO
2,85562,46.17,35.85,10.32,CELINA LEÃO
9,99822,48.01,36.37,11.64,CELINA LEÃO
15,129191,48.29,35.60,12.69,CELINA LEÃO
18,99628,49.00,35.46,13.54,CELINA LEÃO
17,93117,49.79,34.90,14.89,CELINA LEÃO
5,98278,50.61,34.98,15.63,CELINA LEÃO


vencedor
CELINA LEÃO      17
LEANDRO GRASS     2


In [11]:
print("Zona mais favorável a Celina :", duelo.margem_pp.idxmax(), f"({duelo.margem_pp.max():+.1f} p.p.)")
print("Zona mais favorável a Grass  :", duelo.margem_pp.idxmin(), f"({duelo.margem_pp.min():+.1f} p.p.)")
apertada = duelo.margem_pp.abs().idxmin()
print("Zona mais disputada          :", apertada, f"({duelo.margem_pp[apertada]:+.1f} p.p.)")

peso = (duelo.votos_zona / duelo.votos_zona.sum() * 100).sort_values(ascending=False).head(5)
print("\nZonas com maior peso no total de votos (%):"); print(peso.round(1).to_string())

Zona mais favorável a Celina : 13 (+26.1 p.p.)
Zona mais favorável a Grass  : 14 (-10.0 p.p.)
Zona mais disputada          : 1 (-3.5 p.p.)

Zonas com maior peso no total de votos (%):
NR_ZONA
15   7.80
16   6.60
13   6.20
9    6.00
18   6.00


In [12]:
#Desempenho dos Partidos por Cargo

def fragmentacao(base, cargo):
    v = base.groupby("SG_PARTIDO")[VOTOS].sum().sort_values(ascending=False)
    p = v / v.sum()
    
    return v, p * 100, {"cargo": cargo, "partidos": len(v), "maior_partido": p.index[0],
                         "% maior": p.iloc[0]*100, "% top3": p.head(3).sum()*100}

v_dd,  p_dd,  m_dd  = fragmentacao(dd, "Deputado Distrital")
v_fed, p_fed, m_fed = fragmentacao(df_fed, "Deputado Federal")
metricas = pd.DataFrame([m_dd, m_fed]).set_index("cargo")
display(metricas)

,partidos,maior_partido,% maior,% top3
cargo,,,,
Deputado Distrital,24,PL,15.54,36.59
Deputado Federal,26,PL,27.21,55.08


In [13]:
#Concentração de votos nos 10 candidatos mais votados a Deputado Distrital
cand_dd = dd.groupby("NM_URNA_CANDIDATO")[VOTOS].sum().sort_values(ascending=False)
top10 = cand_dd.head(10).sum() / cand_dd.sum() * 100
print(f"{len(cand_dd)} candidatos a Dep. Distrital; os 10 mais votados concentram {top10:.1f}% dos votos.")
display(ranking(dd, 10))

415 candidatos a Dep. Distrital; os 10 mais votados concentram 29.9% dos votos.


,NM_URNA_CANDIDATO,SG_PARTIDO,QT_VOTOS_NOMINAIS_VALIDOS,% votos
1,MAX MACIEL,PSOL,92234,5.67
2,EDUARDO PEDROSA,UNIÃO,50833,3.12
3,CHICO VIGILANTE,PT,50046,3.08
4,JOAQUIM RORIZ NETO,PL,45604,2.80
5,ESTEFANE SAMPAIO,REPUBLICANOS,45558,2.80
6,MÚCIO,PSB,42769,2.63
7,GABRIEL MAGNO,PT,42331,2.60
8,KEKA BAGNO,PSOL,40569,2.49
9,ROOSEVELT VILELA,PL,39181,2.41
10,ROBÉRIO NEGREIROS,PODE,37401,2.30


In [15]:
# Comparecimento e abstenção
det = next((p for p in Path("data").glob("detalhe_votacao_munzona_2026*")), None)
if det is None:
    print("Arquivo de detalhe da votação não encontrado — seção ignorada.")
else:
    d = pd.read_csv(det, sep=";", encoding="latin1") if det.suffix == ".csv" else pd.read_excel(det)
    d = d[d.DS_CARGO == "Governador"]
    z = d.groupby("NR_ZONA")[["QT_APTOS", "QT_COMPARECIMENTO", "QT_ABSTENCOES"]].sum()
    z["comparecimento_%"] = z.QT_COMPARECIMENTO / z.QT_APTOS * 100
    display(z.sort_values("comparecimento_%"))

Arquivo de detalhe da votação não encontrado — seção ignorada.
